# Phase 1: LLAMA + MBPP Dataset Results Inspection

This notebook inspects Phase 1 LLAMA results to verify:
- Dataset files created correctly
- Model used (LLAMA-3.1-8B)
- Prompts formatted correctly
- Code generation working
- Evaluation results (pass/fail)
- Activations captured (31 layers)

In [ ]:
import pandas as pd
import os
from pathlib import Path
import glob
import json
import numpy as np

# Set pandas display options to show FULL content
pd.set_option('display.max_colwidth', None)  # Show full column content
pd.set_option('display.max_columns', None)   # Show all columns
pd.set_option('display.width', None)         # Don't wrap to multiple lines
pd.set_option('display.max_rows', None)      # Show all rows

In [ ]:
# Auto-discovery of Phase 1 LLAMA data
datasets_dir = "../data/phase1_0_llama/"
pattern = os.path.join(datasets_dir, "dataset_*.parquet")
matching_files = sorted(glob.glob(pattern), reverse=True)

if not matching_files:
    raise FileNotFoundError(f"No dataset files found in {datasets_dir}")

latest_file = matching_files[0]
print(f"Using: {Path(latest_file).name}")

In [ ]:
# Check activations directory
activations_dir = Path(datasets_dir) / "activations"

if activations_dir.exists():
    subdirs = [d for d in activations_dir.iterdir() if d.is_dir()]
    for subdir in subdirs:
        files = list(subdir.glob("*.npz"))
        print(f"{subdir.name}/: {len(files)} files")
        if files:
            sample_data = np.load(files[0])
            for key in sample_data.files:
                print(f"  Shape ({key}): {sample_data[key].shape}")
else:
    print(f"Activations directory not found: {activations_dir}")

In [ ]:
# Load dataset
df = pd.read_parquet(latest_file)
print(f"Records: {len(df)}, Columns: {list(df.columns)}")

if 'test_passed' in df.columns:
    n_passed = df['test_passed'].sum()
    print(f"Pass rate: {n_passed}/{len(df)} ({n_passed/len(df)*100:.2f}%)")

In [ ]:
# Display all records
display(df)

In [ ]:
# Sample correct generation
if 'test_passed' in df.columns and df['test_passed'].any():
    correct_sample = df[df['test_passed'] == True].iloc[0]
    print(f"Task ID: {correct_sample.get('task_id', 'N/A')}")
    print(f"Generated Code:\n{correct_sample.get('generated_code', 'N/A')}")

In [ ]:
# Sample incorrect generation
if 'test_passed' in df.columns and (~df['test_passed']).any():
    incorrect_sample = df[df['test_passed'] == False].iloc[0]
    print(f"Task ID: {incorrect_sample.get('task_id', 'N/A')}")
    print(f"Generated Code:\n{incorrect_sample.get('generated_code', 'N/A')}")

In [ ]:
# Summary
print(f"File: {Path(latest_file).name}, Records: {len(df)}")
if 'test_passed' in df.columns:
    print(f"Pass rate: {df['test_passed'].mean()*100:.2f}%")
print(f"Activations exist: {activations_dir.exists()}")